<a href="https://colab.research.google.com/github/gumaruw/Titanic-Veri-Seti-ile-Makine-Ogrenimi/blob/main/titanic_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Kaggle API anahtar dosyasını yükleme
from google.colab import files
files.upload()


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"gumaruw","key":"f68a6a342a937d5dd117a07892257421"}'}

In [ ]:
# Kaggle API anahtar dosyasını doğru yere taşıma
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Kaggle Titanic veri setini indirme
!kaggle datasets download -d heptapod/titanic

!unzip titanic.zip

Dataset URL: https://www.kaggle.com/datasets/heptapod/titanic
License(s): DbCL-1.0
  0% 0.00/10.8k [00:00<?, ?B/s]
100% 10.8k/10.8k [00:00<00:00, 17.2MB/s]
Archive:  titanic.zip
  inflating: train_and_test2.csv     


In [ ]:
# Gerekli kütüphaneleri yükleme
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [ ]:
# Veri setini okuma
df = pd.read_csv('/content/train_and_test2.csv')

print(df.info())
print(df.describe())
print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 28 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Passengerid  1309 non-null   int64  
 1   Age          1309 non-null   float64
 2   Fare         1309 non-null   float64
 3   Sex          1309 non-null   int64  
 4   sibsp        1309 non-null   int64  
 5   zero         1309 non-null   int64  
 6   zero.1       1309 non-null   int64  
 7   zero.2       1309 non-null   int64  
 8   zero.3       1309 non-null   int64  
 9   zero.4       1309 non-null   int64  
 10  zero.5       1309 non-null   int64  
 11  zero.6       1309 non-null   int64  
 12  Parch        1309 non-null   int64  
 13  zero.7       1309 non-null   int64  
 14  zero.8       1309 non-null   int64  
 15  zero.9       1309 non-null   int64  
 16  zero.10      1309 non-null   int64  
 17  zero.11      1309 non-null   int64  
 18  zero.12      1309 non-null   int64  
 19  zero.1

In [ ]:
# Özniteliklerin tanımları
# - PassengerId: Yolcu ID'si
# - Survived: Hayatta kalma (0 = Hayır, 1 = Evet)
# - Pclass: Yolcu sınıfı (1 = 1. sınıf, 2 = 2. sınıf, 3 = 3. sınıf)
# - Name: Yolcunun adı
# - Sex: Cinsiyet
# - Age: Yaş
# - SibSp: Gemideki kardeş/eş sayısı
# - Parch: Gemideki ebeveyn/çocuk sayısı
# - Ticket: Bilet numarası
# - Fare: Bilet ücreti
# - Cabin: Kabin numarası
# - Embarked: Biniş limanı (C = Cherbourg, Q = Queenstown, S = Southampton)

# Veri setindeki ilk birkaç satırı gözden geçirme
print(df.head())

   Passengerid   Age     Fare  Sex  sibsp  zero  zero.1  zero.2  zero.3  \
0            1  22.0   7.2500    0      1     0       0       0       0   
1            2  38.0  71.2833    1      1     0       0       0       0   
2            3  26.0   7.9250    1      0     0       0       0       0   
3            4  35.0  53.1000    1      1     0       0       0       0   
4            5  35.0   8.0500    0      0     0       0       0       0   

   zero.4  ...  zero.12  zero.13  zero.14  Pclass  zero.15  zero.16  Embarked  \
0       0  ...        0        0        0       3        0        0       2.0   
1       0  ...        0        0        0       1        0        0       0.0   
2       0  ...        0        0        0       3        0        0       2.0   
3       0  ...        0        0        0       1        0        0       2.0   
4       0  ...        0        0        0       3        0        0       2.0   

   zero.17  zero.18  2urvived  
0        0        0         0 

In [ ]:
# Kategorik özniteliklerin belirlenmesi
categorical_features = ['Sex', 'Embarked']

# Kategorik özniteliklerin sayısal hale dönüştürülmesi
le = LabelEncoder()
for feature in categorical_features:
    df[feature] = le.fit_transform(df[feature].astype(str))

In [ ]:
# Özniteliklerin gözden geçirilmesi
print(df.head())

   Passengerid   Age     Fare  Sex  sibsp  zero  zero.1  zero.2  zero.3  \
0            1  22.0   7.2500    0      1     0       0       0       0   
1            2  38.0  71.2833    1      1     0       0       0       0   
2            3  26.0   7.9250    1      0     0       0       0       0   
3            4  35.0  53.1000    1      1     0       0       0       0   
4            5  35.0   8.0500    0      0     0       0       0       0   

   zero.4  ...  zero.12  zero.13  zero.14  Pclass  zero.15  zero.16  Embarked  \
0       0  ...        0        0        0       3        0        0         2   
1       0  ...        0        0        0       1        0        0         0   
2       0  ...        0        0        0       3        0        0         2   
3       0  ...        0        0        0       1        0        0         2   
4       0  ...        0        0        0       3        0        0         2   

   zero.17  zero.18  2urvived  
0        0        0         0 

In [ ]:
# Veri setindeki sütun isimlerini kontrol etme
print(df.columns)

Index(['Passengerid', 'Age', 'Fare', 'Sex', 'sibsp', 'zero', 'zero.1',
       'zero.2', 'zero.3', 'zero.4', 'zero.5', 'zero.6', 'Parch', 'zero.7',
       'zero.8', 'zero.9', 'zero.10', 'zero.11', 'zero.12', 'zero.13',
       'zero.14', 'Pclass', 'zero.15', 'zero.16', 'Embarked', 'zero.17',
       'zero.18', '2urvived'],
      dtype='object')


In [ ]:
# Hedef değişkenin ve bağımsız değişkenlerin ayrılması
X = df.drop(columns=['2urvived'])  # Bağımsız değişkenler
y = df['2urvived']  # Hedef değişken

In [ ]:
# Verilerin eğitim ve test olarak bölünmesi
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Normalizasyon
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# Normalize edilmiş verilerin gözden geçirilmesi
print(X_train[:5])

[[ 0.31748678  2.16065705 -0.46750651  1.35550669 -0.48923943  0.
   0.          0.          0.          0.          0.          0.
  -0.42965528  0.          0.          0.          0.          0.
   0.          0.          0.         -0.33130851  0.          0.
   0.60693031  0.          0.        ]
 [-0.28967412  0.19091006 -0.15009468 -0.73773151  0.44209454  0.
   0.          0.          0.          0.          0.          0.
  -0.42965528  0.          0.          0.          0.          0.
   0.          0.          0.         -0.33130851  0.          0.
   0.60693031  0.          0.        ]
 [-0.96311895 -0.59698874 -0.52382151  1.35550669 -0.48923943  0.
   0.          0.          0.          0.          0.          0.
  -0.42965528  0.          0.          0.          0.          0.
   0.          0.          0.          0.85258231  0.          0.
  -0.61981879  0.          0.        ]
 [-1.70284773 -2.01520657 -0.34054178  1.35550669  0.44209454  0.
   0.          0.        

In [ ]:
# KNN için komşuluk değerleri
knn_neighbors = [3, 7, 11]

for k in knn_neighbors:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f'KNN (k={k}) - Accuracy: {accuracy}, Precision: {precision}, Recall: {recall}, F1: {f1}')

KNN (k=3) - Accuracy: 0.8396946564885496, Precision: 0.7246376811594203, Recall: 0.684931506849315, F1: 0.704225352112676
KNN (k=7) - Accuracy: 0.8702290076335878, Precision: 0.8305084745762712, Recall: 0.6712328767123288, F1: 0.7424242424242424
KNN (k=11) - Accuracy: 0.8816793893129771, Precision: 0.875, Recall: 0.6712328767123288, F1: 0.7596899224806202


In [ ]:
# MLP için gizli katman konfigürasyonları
mlp_layers = [(32,), (32, 32), (32, 32, 32)]

for layers in mlp_layers:
    mlp = MLPClassifier(hidden_layer_sizes=layers, max_iter=500, random_state=42)
    mlp.fit(X_train, y_train)
    y_pred = mlp.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f'MLP (layers={layers}) - Accuracy: {accuracy}, Precision: {precision}, Recall: {recall}, F1: {f1}')

/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


MLP (layers=(32,)) - Accuracy: 0.8587786259541985, Precision: 0.8, Recall: 0.6575342465753424, F1: 0.7218045112781954


/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


MLP (layers=(32, 32)) - Accuracy: 0.8587786259541985, Precision: 0.7647058823529411, Recall: 0.7123287671232876, F1: 0.7375886524822695
MLP (layers=(32, 32, 32)) - Accuracy: 0.8396946564885496, Precision: 0.7246376811594203, Recall: 0.684931506849315, F1: 0.704225352112676


In [ ]:
# Naive Bayes için varsayılan parametreler
nb = GaussianNB()
nb.fit(X_train, y_train)
y_pred = nb.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f'Naive Bayes - Accuracy: {accuracy}, Precision: {precision}, Recall: {recall}, F1: {f1}')

Naive Bayes - Accuracy: 0.7938931297709924, Precision: 0.6461538461538462, Recall: 0.5753424657534246, F1: 0.6086956521739131
